In [4]:
# ---- Install packages (run once per project, then comment out) ----
# !pip install pandas numpy matplotlib seaborn scikit-learn pyreadr
# !pip install transformers sentence-transformers  # Install together to avoid version conflicts
# !pip install torch --index-url https://download.pytorch.org/whl/cu121  # GPU-enabled PyTorch
#
# NOTE: First run downloads BERT model from Hugging Face (~440MB).
# This takes 2-5 minutes depending on internet speed.
# After first run, the model loads from local cache instantly.

  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pyreadr-0.5.4-cp314-cp314-win_amd64.whl.metadata (1.4 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.1.1-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ------------- -------------------------- 4.2/12.4 MB 21.7 MB/s eta 0:00:01
   ---------------------------------- ----- 10.7/12.4 MB 27.3 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 27.6 MB/s  0:00:00
Using


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [5]:
# ============================================================
# 16 — NLP for Health Text: From Words to Predictions
# ============================================================
# Purpose: Teach Natural Language Processing (NLP) — how to
#   turn text (clinical notes) into numbers a model can use.
#   We generate synthetic clinical notes from our patient data,
#   then classify them as high-risk vs low-risk using three
#   progressively sophisticated approaches.
#
# What we cover:
#   1. Why text needs different preprocessing than tabular data
#   2. Tokenization — splitting text into pieces
#   3. TF-IDF — the classic bag-of-words representation
#   4. BERT embeddings — deep contextual representations
#   5. Text classification — predicting risk from clinical notes
#
# Kernel: .venv (Python 3.10+)
# GPU: RTX 4060 — CUDA acceleration for BERT
# ============================================================

import pandas as pd          # DataFrames — like R's tibble
import numpy as np           # Numerical arrays — like R's vectors
import matplotlib.pyplot as plt  # Plotting — like R's base graphics
import seaborn as sns        # Statistical plots — like R's ggplot2
import os
import re                    # Regular expressions for text cleaning
import warnings
warnings.filterwarnings('ignore')

# scikit-learn for ML pipeline
from sklearn.model_selection import train_test_split   # like caret::createDataPartition
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF text → numbers
from sklearn.linear_model import LogisticRegression    # classifier on text features
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)

# Hugging Face transformers for BERT
from transformers import AutoTokenizer, AutoModel  # Load pre-trained BERT
import torch                                        # PyTorch for GPU acceleration

# GPU detection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")  # Should print 'cuda' on your RTX 4060

# Output directory
OUT_DIR = "Outputs/NLP_Health_Text"
os.makedirs(OUT_DIR, exist_ok=True)

# Consistent color palette (matches R scripts)
BLUE  = "#2196F3"   # Negative / no disease / low risk
RED   = "#F44336"   # Positive / disease / high risk
GREEN = "#4CAF50"   # Correct prediction

# Plot style
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})
sns.set_style('whitegrid')

print("Imports complete")

ModuleNotFoundError: No module named 'transformers'

In [ ]:
# ============================================================
# SECTION 1: WHY TEXT NEEDS DIFFERENT PREPROCESSING
# ============================================================
#
# In Scripts 01-15 (R), our data was TABULAR:
#   age=67, sex=Male, sbp=142, smoking=Current → numbers/categories
#   ML models work directly on these structured columns.
#
# Clinical notes are UNSTRUCTURED TEXT:
#   "67-year-old male presents with chest pain and elevated BP.
#    History of smoking. Cholesterol 242. Recommend stress test."
#
# The SAME INFORMATION is in both — but text has challenges:
#   1. Variable length (10 words vs 500 words)
#   2. Synonyms ("heart attack" = "myocardial infarction")
#   3. Context matters ("no chest pain" vs "chest pain")
#   4. Typos, abbreviations, jargon
#
# We need to convert text → NUMBERS that capture meaning.
# Three approaches, from simple to sophisticated:
#   TF-IDF: Count words, weight by importance. Fast but ignores order.
#   Word2Vec: Each word → a vector. Captures similarity but not context.
#   BERT: Each word → a vector THAT DEPENDS ON CONTEXT. State of the art.
# ============================================================


# ---- Load Base Data ----
# WHY: We use our existing patient data to GENERATE synthetic clinical notes.
# This way you see the connection between the tabular features you already know
# and their text representation.
try:
    import pyreadr
    ml_data = pyreadr.read_r("Data/ml_teaching_data.rds")[None]
    print("Loaded data from .rds")
except ImportError:
    ml_data = pd.read_csv("Data/ml_teaching_data.csv")
    print("Loaded data from .csv")

print(f"Dataset: {ml_data.shape[0]} patients, {ml_data.shape[1]} columns")


# ---- Generate Synthetic Clinical Notes ----
# WHY: Real clinical notes are protected by HIPAA. We create realistic
# notes from our synthetic patient data. Each note describes a patient
# using natural language, just like a doctor would write.

def generate_clinical_note(row):
    """Generate a synthetic clinical note from a patient's tabular data."""
    
    # Demographics
    age = int(row['age'])
    sex = 'male' if row['sex'] == 'Male' else 'female'
    
    # Smoking description
    smoking_map = {'Never': 'no smoking history',
                   'Former': 'former smoker',
                   'Current': 'active smoker'}
    smoking = smoking_map.get(str(row['smoking_status']), 'unknown smoking status')
    
    # BMI category
    bmi = row['bmi']
    if bmi < 18.5:
        bmi_desc = 'underweight'
    elif bmi < 25:
        bmi_desc = 'normal weight'
    elif bmi < 30:
        bmi_desc = 'overweight'
    else:
        bmi_desc = 'obese'
    
    # Blood pressure
    sbp = int(row['sbp'])
    bp_desc = 'elevated blood pressure' if sbp > 130 else 'normal blood pressure'
    
    # Cholesterol
    chol = int(row['cholesterol'])
    chol_desc = 'elevated cholesterol' if chol > 200 else 'normal cholesterol'
    
    # Glucose
    glucose = int(row['fasting_glucose'])
    glucose_desc = 'elevated fasting glucose' if glucose > 100 else 'normal glucose'
    
    # Disease mentions
    conditions = []
    if row['diabetes'] == 1:
        conditions.append('diabetes mellitus')
    if row['heart_disease'] == 1:
        conditions.append('known cardiovascular disease')
    if row['cancer'] == 1:
        conditions.append('history of malignancy')
    condition_text = ', '.join(conditions) if conditions else 'no significant past medical history'
    
    # Medications
    med_count = int(row['medication_count'])
    med_desc = f'currently on {med_count} medications' if med_count > 0 else 'not currently on medications'
    
    # Assemble the note
    note = (
        f"{age}-year-old {sex} patient presents for routine evaluation. "
        f"Patient is a {smoking}. BMI {bmi:.1f} ({bmi_desc}). "
        f"Vitals show {bp_desc} (SBP {sbp}), {chol_desc} ({chol}), "
        f"and {glucose_desc} ({glucose}). "
        f"Medical history: {condition_text}. "
        f"Patient is {med_desc}."
    )
    
    return note


# Generate notes for all patients
np.random.seed(42)
ml_data['clinical_note'] = ml_data.apply(generate_clinical_note, axis=1)

# Create binary risk label: high-risk = heart_disease OR (age > 70 AND diabetes)
# WHY: This gives us a meaningful classification target that's related to
# the information in the notes. Not identical to heart_disease alone — the
# model has to learn from the TEXT, not just one column.
ml_data['high_risk'] = (
    (ml_data['heart_disease'] == 1) |
    ((ml_data['age'] > 70) & (ml_data['diabetes'] == 1))
).astype(int)

print(f"Clinical notes generated: {len(ml_data)} notes")
print(f"High-risk patients: {ml_data['high_risk'].sum()} ({ml_data['high_risk'].mean()*100:.1f}%)")
print(f"\nExample note (patient 0):\n{ml_data['clinical_note'].iloc[0]}")
print(f"\nExample note (patient 100):\n{ml_data['clinical_note'].iloc[100]}")


# ---- Train/Test Split ----
# WHY: Same logic as R scripts — hold out patients the model never sees.
# stratify= ensures both sets have similar high_risk prevalence.
X_text = ml_data['clinical_note'].values
y = ml_data['high_risk'].values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain: {len(X_train_text)} | Test: {len(X_test_text)}")
print(f"Train prevalence: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%")


# ---- Tokenization Demo ----
# WHAT IS TOKENIZATION?
#   Splitting text into pieces ("tokens"). Different methods:
#   - Word tokenization: split on spaces → ["Patient", "is", "a", "67-year-old"]
#   - Subword tokenization (BERT): split into sub-pieces → ["67", "-", "year", "-", "old"]
#   Subword handles unknown words by breaking them into known pieces.

example_note = X_train_text[0]
print("\n--- Tokenization Demo ---")
print(f"Original text: {example_note[:100]}...")

# Simple word tokenization (like R's str_split)
word_tokens = example_note.lower().split()
print(f"Word tokens ({len(word_tokens)} tokens): {word_tokens[:10]}...")

# BERT subword tokenization
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_tokens = bert_tokenizer.tokenize(example_note.lower())
print(f"BERT tokens ({len(bert_tokens)} tokens): {bert_tokens[:15]}...")
print("\nNotice: BERT splits '67-year-old' differently than word tokenization.")
print("BERT uses 'subword' tokens — it can handle words it's never seen before.")

In [ ]:
# ============================================================
# SECTION 2: TF-IDF — THE CLASSIC APPROACH
# ============================================================
#
# WHAT IS TF-IDF?
#   TF = Term Frequency: how often a word appears in THIS document
#   IDF = Inverse Document Frequency: how rare the word is ACROSS all documents
#   TF-IDF = TF × IDF
#
# WHY IT WORKS:
#   Words like "the", "is", "patient" appear in every note → low IDF → low TF-IDF
#   Words like "diabetes", "malignancy", "elevated" appear only in some → high IDF → high TF-IDF
#   TF-IDF automatically upweights DISTINCTIVE words and downweights common ones.
#
# LIMITATIONS:
#   - Ignores word ORDER: "no chest pain" and "chest pain no" look the same
#   - No understanding of MEANING: "heart attack" and "myocardial infarction" are
#     completely different features, even though they mean the same thing
#   - Each word is independent ("bag of words" assumption)
#
# R EQUIVALENT: tm::DocumentTermMatrix() or tidytext::bind_tf_idf()
# ============================================================

# ---- Fit TF-IDF ----
# max_features=5000: keep only the 5000 most important words
# ngram_range=(1,2): include single words AND two-word phrases
#   This helps capture "chest pain" as one feature instead of separate words
# stop_words='english': remove common words like "the", "is", "and"
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),      # Unigrams + bigrams
    stop_words='english',
    min_df=5,                # Word must appear in at least 5 documents
    max_df=0.95              # Ignore words in >95% of documents
)

# fit_transform on train, transform on test (same as R: fit on train, apply to test)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

print(f"TF-IDF matrix: {X_train_tfidf.shape[0]} documents × {X_train_tfidf.shape[1]} features")
print(f"Sparsity: {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]))*100:.1f}%")
print(f"Top 20 features: {tfidf.get_feature_names_out()[:20].tolist()}")


# ---- Train Classifier on TF-IDF ----
# Logistic regression on TF-IDF features — the classic NLP baseline.
# Like glmnet in R but on word-count features instead of patient measurements.
lr_tfidf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)

tfidf_probs = lr_tfidf.predict_proba(X_test_tfidf)[:, 1]
tfidf_preds = (tfidf_probs >= 0.5).astype(int)
tfidf_auc   = roc_auc_score(y_test, tfidf_probs)

print(f"\nTF-IDF + Logistic Regression — Test AUC: {tfidf_auc:.4f}")
print(classification_report(y_test, tfidf_preds, target_names=['Low Risk', 'High Risk']))


# ---- Top Predictive Words ----
# Which words are most associated with high risk?
feature_names = tfidf.get_feature_names_out()
coefs = lr_tfidf.coef_[0]

top_positive = pd.DataFrame({
    'word': feature_names[np.argsort(coefs)[-15:]],
    'coefficient': np.sort(coefs)[-15:]
}).sort_values('coefficient', ascending=False)

top_negative = pd.DataFrame({
    'word': feature_names[np.argsort(coefs)[:15]],
    'coefficient': np.sort(coefs)[:15]
}).sort_values('coefficient')

print("\nTop words INCREASING risk prediction:")
print(top_positive.to_string(index=False))
print("\nTop words DECREASING risk prediction:")
print(top_negative.to_string(index=False))


# ============================================================
# SECTION 3: BERT EMBEDDINGS — DEEP CONTEXTUAL REPRESENTATIONS
# ============================================================
#
# WHAT IS BERT?
#   BERT (Bidirectional Encoder Representations from Transformers) is
#   a pre-trained language model (see Script 14 for Transformer concepts).
#   It was trained on billions of words of English text and learned to
#   UNDERSTAND language — not just count words.
#
# HOW BERT EMBEDDINGS WORK:
#   1. Feed a text through BERT
#   2. BERT outputs a 768-dimensional vector for each token
#   3. Average all token vectors → one 768-dim vector for the whole text
#   This vector CAPTURES MEANING — similar texts get similar vectors.
#
# WHY BERT IS BETTER THAN TF-IDF:
#   TF-IDF: "no chest pain" looks similar to "chest pain" (same words!)
#   BERT: understands that "no" negates "chest pain" → very different vectors
#
#   TF-IDF: "heart attack" and "myocardial infarction" are unrelated features
#   BERT: knows they mean the same thing → similar vectors
#
# TRADE-OFF:
#   TF-IDF: fast (seconds), no GPU needed, good baseline
#   BERT: slow (minutes), benefits from GPU, better understanding
# ============================================================

# ---- Load Pre-trained BERT ----
# WHY: We don't train BERT from scratch (that would take weeks on a
# supercomputer). We use a pre-trained model and extract embeddings.
# This is called TRANSFER LEARNING — use knowledge learned on general
# text and apply it to our clinical notes.
print("\nLoading BERT model (first run downloads ~440MB, then cached)...")
bert_model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name).to(DEVICE)
bert_model.eval()  # Set to evaluation mode (no training, no dropout)
print(f"BERT loaded on {DEVICE}")


def get_bert_embedding(text, tokenizer, model, max_length=128):
    """Get a 768-dimensional embedding for a text string.
    
    Steps:
    1. Tokenize the text into BERT's subword tokens
    2. Convert tokens to numerical IDs
    3. Feed through BERT to get contextualized representations
    4. Average all token representations → one vector per document
    """
    # Tokenize and convert to tensor
    inputs = tokenizer(
        text,
        return_tensors='pt',      # Return PyTorch tensors
        max_length=max_length,    # Truncate long texts
        truncation=True,
        padding=True
    ).to(DEVICE)
    
    # Get BERT output (no gradient computation needed — we're not training)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # outputs.last_hidden_state shape: (1, n_tokens, 768)
    # Average across tokens to get one vector per document
    # like R's colMeans() but on a 3D tensor
    embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()[0]
    return embedding


# ---- Generate BERT Embeddings ----
# WHY subsample: Embedding 25k documents through BERT takes ~30 minutes.
# We use 3000 train + 1000 test for a manageable demo (~5 minutes on GPU).
N_TRAIN_BERT = 3000
N_TEST_BERT  = 1000

np.random.seed(42)
train_bert_idx = np.random.choice(len(X_train_text), N_TRAIN_BERT, replace=False)
test_bert_idx  = np.random.choice(len(X_test_text), N_TEST_BERT, replace=False)

print(f"Generating BERT embeddings for {N_TRAIN_BERT} train + {N_TEST_BERT} test documents...")
print("(This takes ~3-5 minutes on GPU, ~15-20 on CPU)")

# Process in batches to show progress
X_train_bert = np.zeros((N_TRAIN_BERT, 768))
for i, idx in enumerate(train_bert_idx):
    X_train_bert[i] = get_bert_embedding(X_train_text[idx], tokenizer, bert_model)
    if (i + 1) % 500 == 0:
        print(f"  Train: {i+1}/{N_TRAIN_BERT}")

X_test_bert = np.zeros((N_TEST_BERT, 768))
for i, idx in enumerate(test_bert_idx):
    X_test_bert[i] = get_bert_embedding(X_test_text[idx], tokenizer, bert_model)
    if (i + 1) % 500 == 0:
        print(f"  Test: {i+1}/{N_TEST_BERT}")

y_train_bert = y_train[train_bert_idx]
y_test_bert  = y_test[test_bert_idx]

print(f"BERT embeddings complete: train {X_train_bert.shape}, test {X_test_bert.shape}")


# ---- Train Classifier on BERT Embeddings ----
lr_bert = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_bert.fit(X_train_bert, y_train_bert)

bert_probs = lr_bert.predict_proba(X_test_bert)[:, 1]
bert_preds = (bert_probs >= 0.5).astype(int)
bert_auc   = roc_auc_score(y_test_bert, bert_probs)

print(f"\nBERT + Logistic Regression — Test AUC: {bert_auc:.4f}")
print(classification_report(y_test_bert, bert_preds, target_names=['Low Risk', 'High Risk']))

# Compare TF-IDF on same subset for fair comparison
X_test_tfidf_sub = tfidf.transform(X_test_text[test_bert_idx])
tfidf_probs_sub  = lr_tfidf.predict_proba(X_test_tfidf_sub)[:, 1]
tfidf_auc_sub    = roc_auc_score(y_test_bert, tfidf_probs_sub)

print(f"\nFair comparison (same {N_TEST_BERT} test patients):")
print(f"  TF-IDF AUC: {tfidf_auc_sub:.4f}")
print(f"  BERT AUC:   {bert_auc:.4f}")

In [ ]:
# ============================================================
# SECTION 4: VISUALIZATIONS
# ============================================================

# ---- 4a: ROC Curves — TF-IDF vs BERT ----
fpr_tfidf, tpr_tfidf, _ = roc_curve(y_test_bert, tfidf_probs_sub)
fpr_bert, tpr_bert, _   = roc_curve(y_test_bert, bert_probs)

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot(fpr_tfidf, tpr_tfidf, color=BLUE, linewidth=2,
        label=f'TF-IDF + LogReg (AUC={tfidf_auc_sub:.3f})')
ax.plot(fpr_bert, tpr_bert, color=RED, linewidth=2,
        label=f'BERT + LogReg (AUC={bert_auc:.3f})')
ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — TF-IDF vs BERT Text Classification\n'
             'Same test patients, same classifier, different text representations',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '01_roc_tfidf_vs_bert.png'), dpi=150, bbox_inches='tight')
plt.show()


# ---- 4b: Top TF-IDF Words Bar Chart ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top_positive['word'], top_positive['coefficient'], color=RED)
axes[0].set_title('Words Increasing Risk Prediction', fontweight='bold')
axes[0].set_xlabel('Logistic Regression Coefficient')

axes[1].barh(top_negative['word'], top_negative['coefficient'], color=BLUE)
axes[1].set_title('Words Decreasing Risk Prediction', fontweight='bold')
axes[1].set_xlabel('Logistic Regression Coefficient')

fig.suptitle('Most Predictive Words — TF-IDF + Logistic Regression\n'
             'Which words in clinical notes are most associated with high/low risk?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '02_top_tfidf_words.png'), dpi=150, bbox_inches='tight')
plt.show()


# ---- 4c: BERT Embedding Space (2D Projection) ----
# Use PCA to project 768-dim BERT embeddings to 2D for visualization
# like R's prcomp() — same idea as Script 07
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
bert_2d = pca.fit_transform(X_test_bert)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(bert_2d[:, 0], bert_2d[:, 1],
                     c=y_test_bert, cmap='coolwarm', alpha=0.5, s=20)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('BERT Embeddings in 2D (PCA Projection)\n'
             'Blue = low risk, Red = high risk. Separation = BERT captures risk signal.',
             fontsize=13, fontweight='bold')
plt.colorbar(scatter, label='High Risk', shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '03_bert_embedding_space.png'), dpi=150, bbox_inches='tight')
plt.show()


# ---- 4d: Confusion Matrix — BERT ----
cm = confusion_matrix(y_test_bert, bert_preds)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — BERT Text Classification\n'
             'Same layout as R scripts: rows = actual, columns = predicted',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '04_confusion_matrix_bert.png'), dpi=150, bbox_inches='tight')
plt.show()


# ---- 4e: Method Comparison Bar Chart ----
methods = ['TF-IDF + LogReg', 'BERT + LogReg']
aucs    = [tfidf_auc_sub, bert_auc]
colors  = [BLUE, RED]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, aucs, color=colors, width=0.5)
for bar, auc_val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{auc_val:.4f}', ha='center', fontweight='bold', fontsize=13)
ax.set_ylim(0, 1)
ax.set_ylabel('Test AUC')
ax.set_title('NLP Method Comparison — Text Classification AUC\n'
             'Same clinical notes, same classifier, different text representations',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '05_method_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


# ---- Save Results ----
results = {
    'tfidf_auc': tfidf_auc,
    'bert_auc': bert_auc,
    'tfidf_auc_fair': tfidf_auc_sub,
    'n_train_bert': N_TRAIN_BERT,
    'n_test_bert': N_TEST_BERT
}
pd.DataFrame([results]).to_csv(os.path.join(OUT_DIR, 'results_nlp.csv'), index=False)

# Save the clinical notes for Script 17 (Embeddings & Semantic Search)
ml_data[['id', 'clinical_note', 'high_risk', 'heart_disease', 'age', 'sex',
         'smoking_status', 'bmi', 'sbp', 'cholesterol']].to_csv(
    os.path.join('Data', 'clinical_notes.csv'), index=False
)
print(f"\nClinical notes saved to Data/clinical_notes.csv for Script 17")
print(f"Results saved to {OUT_DIR}")

In [ ]:
# ============================================================
# SECTION 5: SUMMARY — WHAT WE LEARNED
# ============================================================

print("="*60)
print("NLP FOR HEALTH TEXT — SUMMARY")
print("="*60)
print(f"Documents: {len(ml_data)} synthetic clinical notes")
print(f"TF-IDF AUC: {tfidf_auc_sub:.4f}")
print(f"BERT AUC:   {bert_auc:.4f}")
print()
print("WHAT WE LEARNED:")
print("1.  Text data needs fundamentally different preprocessing than tabular data.")
print("    You can't feed raw text to XGBoost — it needs numbers.")
print()
print("2.  TOKENIZATION splits text into pieces. Word tokenization is simple;")
print("    BERT's subword tokenization handles unknown words by splitting them.")
print()
print("3.  TF-IDF converts text to a sparse matrix of word importance scores.")
print("    It's fast and surprisingly effective as a baseline.")
print("    Limitation: ignores word ORDER and MEANING ('no chest pain' ≈ 'chest pain').")
print()
print("4.  BERT EMBEDDINGS capture CONTEXTUAL MEANING — each word's representation")
print("    depends on the surrounding words. 'No chest pain' ≠ 'chest pain'.")
print("    Pre-trained on billions of words, so it already 'understands' language.")
print()
print("5.  TRANSFER LEARNING: We didn't train BERT from scratch (impossible without")
print("    a supercomputer). We used a pre-trained model and extracted features.")
print("    This is like using a pre-built Random Forest as a feature extractor.")
print()
print("6.  For our synthetic notes, TF-IDF does well because the notes are")
print("    template-based and keywords map directly to risk. On REAL clinical notes")
print("    with varied language, BERT would show a bigger advantage.")
print()
print("WHEN TO USE EACH:")
print("  TF-IDF:  Fast baseline, small datasets, simple text")
print("  BERT:    Complex text, negation matters, synonyms matter, have GPU")
print()
print("HOW THIS CONNECTS TO OTHER SCRIPTS:")
print("  Scripts 01-15 (R): Tabular data — numbers in columns.")
print("  This script: Text data — words in sentences. Same goal (classify),")
print("  very different input. NLP is the bridge from structured to unstructured data.")
print("  Script 17: Takes the embeddings concept further — semantic search.")
print("  Script 18: Uses LLMs (GPT) to GENERATE text, not just classify it.")
print("="*60)
print("\nScript 16 complete.")